# 01 - PDF Extraction + Quality Assessment
Ekstraksi teks dari PDF publikasi BPS menggunakan PyMuPDFLLM .

In [1]:
!pip install pymupdf4llm pdfplumber -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.3/77.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 68.2 MB/s eta 0:00:00


In [2]:
import importlib.metadata
import io
import json
import re
from collections import Counter
from pathlib import Path
import pandas as pd
import pdfplumber
import pymupdf4llm

## 1. Konfigurasi

In [3]:
INPUT_DIR  = Path("/kaggle/input/datasets/muhammadnabilul/bps-publications")
df_meta    = pd.read_excel(INPUT_DIR / 'List_publikasi.xlsx')
PDF_DIR    = INPUT_DIR / "pdf"
OUT_DIR    = Path("/kaggle/working/extracted")
OUT_DIR.mkdir(exist_ok=True)

## Load Metadata

In [4]:
df_meta.columns = ['tahun', 'judul', 'bps_url', 'nama_file']
df_meta['tahun'] = df_meta['tahun'].astype(str)
metadata_list = df_meta.to_dict(orient='records')
 
print(f'Total dokumen: {len(df_meta)}')
for i, row in df_meta.iterrows():
    print(f'  [{i+1:>3}] ({row["tahun"]}) {row["judul"]}')

Total dokumen: 47
  [  1] (2023) Statistik Indonesia 2023
  [  2] (2023) Indeks Pembangunan Manusia 2022
  [  3] (2023) Indeks Perilaku Anti Korupsi 2023
  [  4] (2023) Indikator Kesejahteraan Rakyat 2023
  [  5] (2023) Kesejahteraan Anak Indonesia: Analisis Deprivasi Hak Anak Multidimensi 2022
  [  6] (2023) Kesejahteraan Anak Indonesia: Analisis Kemiskinan Anak Moneter 2022
  [  7] (2023) Laporan Indeks Khusus Penanganan Stunting 2021–2022
  [  8] (2023) Laporan Indeks Khusus Penanganan Stunting Kabupaten/Kota 2021–2022
  [  9] (2023) Profil Anak Usia Dini 2023
  [ 10] (2023) Profil Statistik Kesehatan 2023
  [ 11] (2023) Statistik Kesehatan 2022
  [ 12] (2023) Statistik Kesejahteraan Rakyat 2023
  [ 13] (2023) Statistik Kriminal 2023
  [ 14] (2023) Statistik Pemuda Indonesia 2023
  [ 15] (2023) Statistik Pendidikan 2023
  [ 16] (2023) Statistik Penduduk Lanjut Usia 2023
  [ 17] (2023) Statistik Politik 2023
  [ 18] (2024) Statistik Indonesia 2024
  [ 19] (2024) Indeks Pembangunan Ma

## 2. Bahan Ekstraksi

In [5]:
EXPERIMENT_NAME = "(1) no_ocr_pdfplumber"

TABLE_ENGINE = "pdfplumber"

EXTRACTION_CONFIG = {
    "page_chunks"  : True,
    "force_text"   : True,
    "ignore_code"  : False,
    "embed_images" : False,
    "write_images" : False,
    "use_ocr"      : False,
    "force_ocr"    : False,
    "show_progress": True,
}

## Fungsi Ekstraksi
Selengkapnya bisa dilihat di https://pymupdf.readthedocs.io/en/latest/pymupdf4llm/api.html

In [6]:
def extract_tables_pdfplumber(pdf_path: Path) -> dict[int, list[dict]]:
    page_tables = {}
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages, start=1):
                tables_out = []
                try:
                    tables = page.extract_tables()
                    for t_idx, table in enumerate(tables):
                        if not table:
                            continue
                        rows_clean = [
                            [
                                str(cell).strip()
                                if cell is not None else ""
                                for cell in row
                            ]
                            for row in table
                        ]
                        tables_out.append({
                            "table_index": t_idx,
                            "row_count": len(rows_clean),
                            "col_count": (
                                max(len(r) for r in rows_clean)
                                if rows_clean else 0
                            ),
                            "data": rows_clean,
                        })
                except Exception:
                    pass
                page_tables[page_num] = tables_out
    except Exception:
        pass
    return page_tables

In [7]:
def extract_pdf(pdf_path: Path, row: pd.Series) -> tuple[list, dict]:

    # Ekstrak Teks
    chunks = pymupdf4llm.to_markdown(
        str(pdf_path),
        **EXTRACTION_CONFIG,
    )

    # Ekstrak Tabel
    page_tables = extract_tables_pdfplumber(
        pdf_path
    )

    pages = []
    flagged_pages = []

    # Looping
    for chunk in chunks:
        metadata    = chunk.get("metadata", {})
        pg_num      = (
            metadata.get("page_number")
            or metadata.get("page")
            or 0
        )
        text        = chunk.get("text", "")
        n_chars     = len(text)
        n_words     = len(text.split())
        page_boxes  = chunk.get("page_boxes",[])
        box_classes = list({b.get("class", "unknown") for b in page_boxes})
        tables      = page_tables.get(pg_num,[])

        # Tanda
        flags = []
        if n_chars == 0:
            flags.append("empty")
        if 0 < n_chars < 100:
            flags.append("low_char")
        if len(chunk.get("images", [])) > 0 and n_chars < 200:
            flags.append("image_dominant")

        # Isian 
        page_entry = {
            "page_number"  : pg_num,
            "chunk_id"     : f"{pdf_path.stem}__p{pg_num:04d}",
            "text_md"      : text,
            "tables"       : tables,
            "page_boxes"   : page_boxes,
            "toc_items"    : chunk.get("toc_items",  []),
            "flags"        : flags,
            "raw_metrics"  : {
                "char_count"    : n_chars,
                "token_estimate": n_chars // 4,
                "line_count"    : len(text.splitlines()),
                "word_count"    : n_words,
                "table_count"   : len(tables),
                "image_count"   : len(chunk.get("images",[])),
                "box_classes"   : box_classes,
                "has_header_footer": has_header_footer,
        	},
        }
        
        pages.append(page_entry)
        
        if flags:
            flagged_pages.append({
                "page_number": pg_num,
                "flags"      : flags,
            })
            
    # Metric doc
    total_halaman  = len(pages)
    total_chars    = sum(p["raw_metrics"]["char_count"]   for p in pages)
    total_words    = sum(p["raw_metrics"]["word_count"]   for p in pages)
    total_tables   = sum(p["raw_metrics"]["table_count"]  for p in pages)
    total_images   = sum(p["raw_metrics"]["image_count"]  for p in pages)
    n_empty        = sum(1 for p in pages if "empty" in p["flags"])
    coverage_ratio = (total_halaman - n_empty) / total_halaman if total_halaman else 0
 
    if coverage_ratio >= 0.70:
        overall_quality = "good"
    elif coverage_ratio >= 0.50:
        overall_quality = "review"
    else:
        overall_quality = "poor"

    # Kualitas
    quality = {
        "total_halaman"           : total_halaman,
        "halaman_diekstrak"       : total_halaman - n_empty,
        "halaman_empty"           : n_empty,
        "coverage_ratio"          : round(coverage_ratio, 3),
        "flagged_pages"           : flagged_pages,
        "n_flagged"               : len(flagged_pages),
        "overall_quality"         : overall_quality,
        "doc_stats": {
            "total_chars" : total_chars,
            "total_words" : total_words,
            "total_tables": total_tables,
            "total_images": total_images,
        },

        # Info
        "experiment"  : EXPERIMENT_NAME,
        "engine"      : "pymupdf4llm",
        "table_engine": TABLE_ENGINE,
        "config"      : EXTRACTION_CONFIG,
        "versions"    : {
             "pymupdf"    : importlib.metadata.version("pymupdf"),
             "pymupdf4llm": importlib.metadata.version("pymupdf4llm"),
             "pdfplumber" : importlib.metadata.version("pdfplumber"),
                },
        }
 
    return pages, quality

## 3. Main Program

In [8]:
all_results     = []
quality_reports = []
total           = len(df_meta)
 
for idx, row in df_meta.iterrows():
    tahun     = str(row['tahun'])
    nama_file = row['nama_file']
    judul     = row['judul']
    pdf_path  = PDF_DIR / tahun / nama_file
 
    print(f'[{idx+1}/{total}] {judul[:60]}')
 
    if not pdf_path.exists():
        print(f'  ✗ File tidak ditemukan: {pdf_path}')
        quality_reports.append({
            'nama_file': nama_file, 'judul': judul, 'tahun': tahun,
            'recommendation': 'file-missing', 'overall_quality': 'poor',
        })
        continue
 
    try:
        pages, quality = extract_pdf(pdf_path, row)
    except Exception as e:
        print(f'  ✗ Error: {e}')
        quality_reports.append({
            'nama_file': nama_file, 'judul': judul, 'tahun': tahun,
            'recommendation': 'error', 'overall_quality': 'poor', 'error': str(e),
        })
        continue
 
    quality.update({'nama_file': nama_file, 'judul': judul, 'tahun': tahun})
    quality_reports.append(quality)
 
    doc_result = {
        'judul'    : judul,
        'tahun'    : tahun,
        'nama_file': nama_file,
        'bps_url'  : row.get('bps_url', ''),
        'n_halaman': quality['total_halaman'],
        'n_chunks' : len(pages),
        'quality'  : quality['overall_quality'],
        'pages'    : pages,
    }
    all_results.append(doc_result)
    
    fname = nama_file.replace('.pdf', '.json')
    
    with open(OUT_DIR / fname, 'w', encoding='utf-8') as f:
        json.dump(doc_result, f, ensure_ascii=False, indent=2)
 
    with open(OUT_DIR / 'quality_report.json', 'w', encoding='utf-8') as f:
        json.dump(quality_reports, f, ensure_ascii=False, indent=2)
 
    print(f'  ✓ {quality["halaman_diekstrak"]}/{quality["total_halaman"]} hal'
          f' | {quality["overall_quality"]}')
 
    if quality['flagged_pages']:
        print(f'  ⚑ {quality["n_flagged"]} halaman di-flag')
 
print(f'\nSelesai: {len(all_results)} dokumen')

[1/47] Statistik Indonesia 2023
Parsing 830 pages of '/kaggle/input/datasets/muhammadnabilul/bps-publications/pdf/2023/statistik-indonesia-2023.pdf'...


100%|██████████| 830/830 [06:15<00:00,  2.21it/s]


Generating markdown text...


100%|██████████| 830/830 [00:00<00:00, 4057.03it/s]


  ✗ Error: name 'has_header_footer' is not defined
[2/47] Indeks Pembangunan Manusia 2022
Parsing 180 pages of '/kaggle/input/datasets/muhammadnabilul/bps-publications/pdf/2023/indeks-pembangunan-manusia-2022.pdf'...


...

  ✗ Error: name 'has_header_footer' is not defined
[47/47] Statistik Sosial Budaya 2024
Parsing 228 pages of '/kaggle/input/datasets/muhammadnabilul/bps-publications/pdf/2025/statistik-sosial-budaya-2024.pdf'...


100%|██████████| 228/228 [01:53<00:00,  2.00it/s]


Generating markdown text...


100%|██████████| 228/228 [00:00<00:00, 6678.08it/s]


  ✗ Error: name 'has_header_footer' is not defined

Selesai: 0 dokumen
